# Загрузка первой общей выгрузки

Набор **first-export-2026-10-21-v1**: 238 уникальных вакансий, 240 включений,
18 срезов. Даты наблюдений — 21.10.2026 по Москве; точный интервал находится
в паспорте. Это разведочная выборка, не готовый сопоставимый месячный таргет.

Notebook только загружает и проверяет данные. Сайт не открывается, обучение не запускается.


In [ ]:
from pathlib import Path
import sys, json, hashlib
from collections import Counter

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "Scraper/runs/first_export_2026-10-21/manifest.json").exists())
sys.path.insert(0, str(ROOT))
from utils.load_dataset import load_dataset

PASSPORT = "Scraper/runs/first_export_2026-10-21/manifest.json"
EXPECTED_PASSPORT_SHA256 = "f514616fbb9014447d217e6d33c5410fd1529e2332acc74b600bf3ecd0eeacf2"
actual_passport_sha256 = hashlib.sha256((ROOT / PASSPORT).read_bytes()).hexdigest()
assert actual_passport_sha256 == EXPECTED_PASSPORT_SHA256, "Паспорт изменился; выберите версию явно"
rows, memberships, passport = load_dataset(PASSPORT, ROOT)
assert passport["dataset_id"] == "first-export-2026-10-21-v1"
print(json.dumps({"dataset_id": passport["dataset_id"],
                  "passport_sha256": hashlib.sha256((ROOT / PASSPORT).read_bytes()).hexdigest(),
                  "unique_vacancies": len(rows), "accepted_memberships": len(memberships),
                  "observation_window": passport["observation_window"],
                  "training_performed": False}, ensure_ascii=False, indent=2))


In [ ]:
coverage = Counter((x["profession"], x["city"]) for x in memberships)
for (profession, city), count in sorted(coverage.items()):
    print(f"{profession} | {city}: {count}")
assert len(coverage) == 18
print("SHA-256, колонки, уникальные ID и покрытие срезов проверены.")


## Продолжение работы

`rows` содержит уникальные вакансии; `memberships` — только принятые включения
основного плана, включая разрешённые замены из резерва. Очередь observations.csv
содержит 795 строк и не является 795 обучающими примерами.

Сумма, период ставки, налоги, нагрузка и условия премий различаются. Сначала
согласуйте целевую переменную и правила сопоставимости. Не выводите месячную
зарплату из частоты выплат и не переносите зарплатные фразы в признаки для
предсказания той же зарплаты. Для проверки закрепите ID и правило разделения,
учитывая повторы между срезами. Обучение и модель в этом примере отсутствуют.

Эксперимент: `ML/experiments/<experiment_id>/`; модель и обработка признаков:
`ML/models/<experiment_id>/`. Начните с `ML/experiment.template.json`.
